# Multitask Stochastic D-Optimal Design

This notebook runs the stochastic D-optimal design training experiments using preprocessed Ames Housing artifacts produced by `05 Multitask D_optimal Design Preprocessing.ipynb`. The data cleaning, clustering, and task design matrix construction are intentionally kept out of this training notebook.


## 1. Verification of the corrected basis setup

Let $d$ be the number of design points and let $q$ be the number of basis functions used for each component of $\theta^*(\lambda)$. We sample one shared set of $q$ random directions in $\mathbb{R}^S$, take their inner products with $\lambda$, and apply a softmax across those $q$ values. This gives a basis vector
$$
\phi(\lambda) \in \mathbb{R}^q,
\qquad
\sum_{k=1}^q \phi_k(\lambda)=1,
\qquad
\phi_k(\lambda) > 0.
$$
The same basis vector is reused for every design point, so each row block of $\Phi(\lambda)$ uses the same $q$ basis values while retaining its own coefficient block $\beta_i$.

Stacking these componentwise bases gives the corrected block-diagonal matrix
$$
\Phi(\lambda) \in \mathbb{R}^{d \times qd},
$$
with one $1 \times q$ block per row. Writing
$$
\beta = (\beta_1^\top, \dots, \beta_d^\top)^\top,
\qquad
\beta_i \in \mathbb{R}^q,
\qquad
p = qd,
$$
we obtain the raw positive weights
$$
\alpha_i(\lambda,\beta) = \phi(\lambda)^\top \beta_i.
$$

Because this corrected block-diagonal parameterization does not preserve the simplex automatically, the notebook converts the raw positive weights into a feasible design by normalizing them:
$$
\theta_i(\lambda,\beta) = \frac{\alpha_i(\lambda,\beta)}{\sum_{r=1}^d \alpha_r(\lambda,\beta)}.
$$
Thus $\theta(\lambda,\beta) \in \Delta_d$.

For task $s$,
$$
M^{(s)}(\theta)=X^{(s)\top}\operatorname{Diag}(\theta)X^{(s)}.
$$
The stochastic objective remains
$$
F_\gamma(\beta)
=
\mathbb{E}_{\lambda\sim\mathbb{P}_\lambda}\left[
\sum_{s=1}^S \lambda_s\big(-\log\det M^{(s)}(\theta(\lambda,\beta))\big)
\right] + \gamma\psi(\beta),
\qquad
\psi(\beta) = -\sum_{j=1}^{qd}\log \beta_j.
$$

The task-level matrix must still be indexed correctly:
$$
A^{(s)}(\lambda,\beta)
:=
X^{(s)}\big(M^{(s)}(\theta(\lambda,\beta))\big)^{-1}X^{(s)\top}.
$$
If
$$
g_\theta = \nabla_\theta \sum_{s=1}^S \lambda_s\big(-\log\det M^{(s)}(\theta)\big),
$$
then
$$
(g_\theta)_i = -\sum_{s=1}^S \lambda_s\,\operatorname{diag}(A^{(s)}(\lambda,\beta))_i.
$$
Since $\theta = \alpha / (\mathbf{1}^\top \alpha)$, the chain rule gives
$$
\nabla_\alpha f = \frac{g_\theta - (\theta^\top g_\theta)\mathbf{1}}{\mathbf{1}^\top\alpha},
$$
and therefore the gradient with respect to the block coefficients is
$$
\frac{\partial f}{\partial \beta_{ik}} = \phi_{ik}(\lambda)\,(\nabla_\alpha f)_i.
$$

The mirror-descent update with the log-barrier reference function is unchanged in form:
$$
\beta_{t+1}
=
\arg\min_{\beta\in\Delta_{qd}}
\Big\{\eta\nabla f_\gamma(\beta_t,\lambda_t)^\top(\beta-\beta_t)+D_\psi(\beta,\beta_t)\Big\},
$$
which again reduces to the scalar root solve
$$
\beta_{t+1,j} = \frac{1}{v_{t,j}+\xi_t},
\qquad
\sum_{j=1}^{qd}\frac{1}{v_{t,j}+\xi_t}=1.
$$

As before, I use
$$
\lambda \sim \operatorname{Dirichlet}(1,\ldots,1).
$$

In [ ]:
import math
import json
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd

SEED = 7
rng = np.random.default_rng(SEED)


In [ ]:
# Optimization hyperparameters.
q = 20
ridge = 1e-5
gamma = 1e-2
eta = .5/(1 + gamma)
T = 1000
n_eval = 1000
checkpoint_interval = 100


In [ ]:
# Load the preprocessed artifacts produced by `05 Multitask D_optimal Design Preprocessing.ipynb`.
preprocessed_dir = "preprocessed"
community_means_csv = f"{preprocessed_dir}/multitask_community_means.csv"
community_sizes_csv = f"{preprocessed_dir}/multitask_community_sizes.csv"
design_matrices_npz = f"{preprocessed_dir}/multitask_design_matrices.npz"
metadata_json = f"{preprocessed_dir}/multitask_preprocessing_metadata.json"

try:
    preprocessing_metadata = json.loads(open(metadata_json, encoding="utf-8").read())
    selected = pd.read_csv(community_means_csv)
    community_sizes = pd.read_csv(community_sizes_csv)["cluster_size"].to_numpy(dtype=int)
    design_data = np.load(design_matrices_npz, allow_pickle=True)
except FileNotFoundError as exc:
    raise FileNotFoundError(
        "Run `05 Multitask D_optimal Design Preprocessing.ipynb` before running this training notebook."
    ) from exc

task_names = [str(task) for task in design_data["task_names"]]
X_list = [design_data[f"X_{task}"] for task in task_names]
S = len(task_names)
d = X_list[0].shape[0]
p = q * d

if len(selected) != d:
    raise ValueError("Community means and design matrices disagree on the number of design points.")
if any(X.shape[0] != d for X in X_list):
    raise ValueError("At least one task design matrix has the wrong number of design points.")

print(f"Loaded preprocessed design points: {d}")
print(f"Cluster size range: {community_sizes.min()} to {community_sizes.max()}")
print(f"Basis dimension q = {q}")
print(f"Parameter dimension p = qd = {p}")
print(f"Tasks: {task_names}")
selected.head()


In [ ]:
# Summary of the loaded task-specific design matrices.
pd.DataFrame(
    {
        "task": task_names,
        "n_design_points": [X.shape[0] for X in X_list],
        "n_features_with_intercept": [X.shape[1] for X in X_list],
    }
)


In [ ]:
# Corrected block-diagonal basis construction with shared directions.
# basis_directions[k, :] is the k-th random direction shared by all design points.
basis_directions = rng.normal(scale=1.0, size=(q, S))


def softmax_1d(scores):
    shifted = scores - scores.max()
    exp_shifted = np.exp(shifted)
    return exp_shifted / exp_shifted.sum()


def sample_lambda(rng):
    return rng.dirichlet(np.ones(S))

# repeat the same row of q basis vectors d times, result is d by q matrix
def make_block_basis(lam):
    phi = softmax_1d(basis_directions @ lam)
    return np.tile(phi, (d, 1))


*How `solve_xi` works:*

The mirror-descent update has the closed form
$$
\beta_{t+1,j} = \frac{1}{v_j + \xi}.
$$
The unknown scalar `\xi` is chosen so that `\beta_{t+1}` stays in the simplex, which means
$$
\sum_{j=1}^p \frac{1}{v_j + \xi} = 1,
\qquad \xi > -\min_j v_j.
$$
`solve_xi` finds this unique root by bisection: near `-\min_j v_j` the left-hand side is very large, while for large `\xi` it is below `1`, so the root is bracketed and then refined until convergence.


In [ ]:
# Objective, gradient, and mirror-descent update for the corrected Phi(lambda).
def psi(beta):
    return -np.sum(np.log(beta))


def grad_psi(beta):
    return -1.0 / beta


def theta_from(beta, lam):
    beta_blocks = beta.reshape(d, q) # make beta into d by q matrix
    phi = make_block_basis(lam)
    alpha = np.sum(phi * beta_blocks, axis=1) # element-wise multiplication, then sum over rows
    alpha = np.clip(alpha, 1e-12, None)
    theta = alpha / alpha.sum()
    return theta, alpha, phi


def information_matrix(X, theta):
    return X.T @ (theta[:, None] * X) + ridge * np.eye(X.shape[1])


def sample_loss_and_grad(beta, lam):
    theta, alpha, phi = theta_from(beta, lam)
    loss = 0.0
    grad_theta = np.zeros(d)

    for s, X in enumerate(X_list):
        M = information_matrix(X, theta)
        sign, logdet = np.linalg.slogdet(M)
        if sign <= 0:
            raise RuntimeError("Encountered a non-positive determinant.")
        loss += lam[s] * (-logdet)

        M_inv = np.linalg.inv(M)
        XM_inv = X @ M_inv
        diag_A_s = np.sum(XM_inv * X, axis=1)
        grad_theta += lam[s] * (-diag_A_s)

    alpha_sum = alpha.sum()
    centered_grad = grad_theta - np.dot(theta, grad_theta)
    grad_alpha = centered_grad / alpha_sum
    grad_beta = (phi * grad_alpha[:, None]).reshape(-1)
    return loss, grad_beta


eval_lambdas = np.array([sample_lambda(rng) for _ in range(n_eval)])


def estimated_stochastic_objective(beta):
    values = [sample_loss_and_grad(beta, lam)[0] for lam in eval_lambdas]
    no_reg = float(np.mean(values))
    return no_reg, no_reg + float(gamma * psi(beta))


def solve_xi(v, tol=1e-12, max_iter=200):
    lower = -np.min(v) + 1e-12
    upper = max(1.0, lower + 1.0)

    def root_fn(xi):
        return np.sum(1.0 / (v + xi)) - 1.0

    while root_fn(upper) > 0:
        upper *= 2.0

    for _ in range(max_iter):
        mid = 0.5 * (lower + upper)
        if root_fn(mid) > 0:
            lower = mid
        else:
            upper = mid
        if upper - lower < tol:
            break

    return 0.5 * (lower + upper)


def mirror_step(beta, grad_f):
    v = eta * grad_f - (1.0 - eta * gamma) * grad_psi(beta)
    xi = solve_xi(v)
    beta_next = 1.0 / (v + xi)
    beta_next = np.clip(beta_next, 1e-15, None)
    beta_next = beta_next / beta_next.sum()
    return beta_next

In [ ]:
# Run stochastic reference function descent under the corrected basis map.
# beta = np.full(p, 1.0 / p)
beta = np.full(p, 1e-3)
beta[0] = 1 - p * 1e-3

history = []
for t in range(T):
    lam_t = sample_lambda(rng)
    sample_loss, grad_f = sample_loss_and_grad(beta, lam_t)
    no_reg_obj, reg_obj = estimated_stochastic_objective(beta)

    history.append(
        {
            "t": t,
            "objective_estimate": no_reg_obj,
            "reg_objective_estimate": reg_obj,
            "sample_loss": sample_loss,
            "regularized_sample_loss": sample_loss + gamma * psi(beta),
            "beta_min": beta.min(),
            "beta_max": beta.max(),
        }
    )

    beta = mirror_step(beta, grad_f)

history = pd.DataFrame(history)

stochastic_reference_history_csv = "stochastic_reference_history.csv"
history.to_csv(stochastic_reference_history_csv, index=False)

history.head()


In [ ]:
summary = pd.DataFrame(
    {
        "metric": [
            "initial objective estimate",
            "final objective estimate",
            "final beta min",
            "final beta max",
        ],
        "value": [
            history["objective_estimate"].iloc[0],
            history["objective_estimate"].iloc[-1],
            beta.min(),
            beta.max(),
        ],
    }
)
summary


The stochastic reference history is saved to `stochastic_reference_history.csv`. Plotting for this and the downstream experiments lives in `06 Multitask D_optimal Design Plots.ipynb`.


In [ ]:
history.loc[:, ['t', 'objective_estimate', 'reg_objective_estimate']].tail(10)


## 2. Training implementation notes

This notebook now starts from saved preprocessing artifacts. Run `05 Multitask D_optimal Design Preprocessing.ipynb` first to produce:
- `preprocessed/multitask_community_means.csv`
- `preprocessed/multitask_community_sizes.csv`
- `preprocessed/multitask_design_matrices.npz`
- `preprocessed/multitask_preprocessing_metadata.json`

Training then loads `X_list`, `task_names`, `d`, `S`, and `p` from those artifacts and runs the stochastic mirror-descent experiments.


## 3. Downstream Task: Comparing Fixed and Adaptive $\gamma$ Schedules

This section leaves the earlier experiment untouched and adds two downstream comparisons.

1. A fixed-`\gamma` sweep using the same initialization for `\beta`.
2. An adaptive-`\gamma` run that starts from a configurable `adaptive_gamma_start`, detects saturation with either a rolling-average gradient detector or a rolling-average unregularized training-objective detector, then updates `\gamma <- adaptive_gamma_shrink * \gamma`.

Both adaptive detectors compare adjacent rolling windows, such as iterations 1-5 against 2-6 when the window is 5. The gradient detector averages the raw stochastic gradients and thresholds the norm of the rolling-average gradient difference. The objective detector thresholds the absolute rolling-average difference of the unregularized training objective. The saturation threshold is scaled as `threshold_ref * (current_gamma / gamma_start) ** alpha`, so positive `alpha` makes the rule more relaxed for larger `gamma` and tighter after each shrink.

Each experiment is saved to CSV. The final comparison plot reads those CSV files back from disk and overlays all unregularized stochastic objective traces.


In [ ]:
# Helper functions for the downstream gamma-comparison experiments.
gamma_values = np.array([1.0, 0.1, 0.01, 0.001])
gamma_sweep_csv = "gamma_sweep_results_vertex.csv"
adaptive_gamma_csv = "adaptive_gamma_results_vertex.csv"
adaptive_gamma_beta_snapshots_npz = "adaptive_gamma_beta_snapshots_vertex.npz"
adaptive_gamma_worst_case_gap_csv = "adaptive_gamma_worst_case_gaps_vertex.csv"
gamma_sweep_beta_snapshots_npz = "gamma_sweep_beta_snapshots_vertex.npz"
gamma_sweep_worst_case_gap_csv = "gamma_sweep_worst_case_gaps_vertex.csv"

# Worst-case gap controls. `checkpoint_interval` is defined in the hyperparameter cell.
worst_case_gap_enabled = True
use_ipopt_for_worst_case_gap = True
worst_case_num_lambda_starts = 32
worst_case_num_local_starts = 6
worst_case_outer_max_iters = 40
worst_case_theta_max_iters = 250


def eta_for_gamma(gamma_value):
    return 0.5 / (1.0 + gamma_value)


def estimated_stochastic_objective_for_gamma(beta, gamma_value):
    values = [sample_loss_and_grad(beta, lam)[0] for lam in eval_lambdas]
    no_reg = float(np.mean(values))
    reg = no_reg + float(gamma_value * psi(beta))
    return no_reg, reg


def mirror_step_for_gamma(beta, grad_f, gamma_value):
    eta_value = eta_for_gamma(gamma_value)
    v = eta_value * grad_f - (1.0 - eta_value * gamma_value) * grad_psi(beta)
    xi = solve_xi(v)
    beta_next = 1.0 / (v + xi)
    beta_next = np.clip(beta_next, 1e-15, None)
    beta_next = beta_next / beta_next.sum()
    return beta_next


def checkpoint_due(epoch_index, total_epochs, interval):
    return epoch_index == 0 or epoch_index == total_epochs or epoch_index % interval == 0


def run_fixed_gamma_experiment(gamma_value, beta_init, lambda_sequence, checkpoint_interval=100):
    beta = beta_init.copy()
    rows = []
    snapshots = [{"gamma": float(gamma_value), "epoch_index": 0, "beta": beta.copy()}]
    total_epochs = len(lambda_sequence)

    for t, lam_t in enumerate(lambda_sequence):
        sample_loss, grad_f = sample_loss_and_grad(beta, lam_t)
        no_reg_obj, reg_obj = estimated_stochastic_objective_for_gamma(beta, gamma_value)
        rows.append({
            "experiment": f"fixed gamma={gamma_value:g}",
            "mode": "fixed",
            "gamma": gamma_value,
            "phase": np.nan,
            "phase_iteration": np.nan,
            "t": t,
            "objective_estimate": no_reg_obj,
            "reg_objective_estimate": reg_obj,
            "sample_loss": sample_loss,
            "eta": eta_for_gamma(gamma_value),
            "beta_min": beta.min(),
            "beta_max": beta.max(),
        })
        beta = mirror_step_for_gamma(beta, grad_f, gamma_value)
        completed_epochs = t + 1
        if checkpoint_due(completed_epochs, total_epochs, checkpoint_interval):
            snapshots.append({
                "gamma": float(gamma_value),
                "epoch_index": completed_epochs,
                "beta": beta.copy(),
            })

    return pd.DataFrame(rows), snapshots


def save_fixed_gamma_beta_snapshots(snapshots, path):
    path = Path(path)
    snapshot_gammas = np.array([item["gamma"] for item in snapshots], dtype=float)
    snapshot_epochs = np.array([item["epoch_index"] for item in snapshots], dtype=int)
    snapshot_betas = np.vstack([item["beta"] for item in snapshots])
    np.savez_compressed(
        path,
        gamma=snapshot_gammas,
        epoch_index=snapshot_epochs,
        beta=snapshot_betas,
    )
    return path


def load_fixed_gamma_beta_snapshots(path):
    data = np.load(path)
    return [
        {
            "gamma": float(gamma_value),
            "epoch_index": int(epoch_index),
            "beta": beta.astype(float),
        }
        for gamma_value, epoch_index, beta in zip(data["gamma"], data["epoch_index"], data["beta"])
    ]


def project_to_simplex(x):
    x = np.asarray(x, dtype=float)
    if x.ndim != 1:
        raise ValueError("Simplex projection expects a one-dimensional vector.")
    u = np.sort(x)[::-1]
    cssv = np.cumsum(u) - 1.0
    ind = np.arange(1, len(x) + 1)
    cond = u - cssv / ind > 0
    if not np.any(cond):
        return np.full_like(x, 1.0 / len(x))
    rho = ind[cond][-1]
    tau = cssv[cond][-1] / rho
    return np.maximum(x - tau, 0.0)


def d_optimal_objective_and_grad_theta(theta, lam):
    objective = 0.0
    grad_theta = np.zeros(d)
    for s, X in enumerate(X_list):
        M = information_matrix(X, theta)
        sign, logdet = np.linalg.slogdet(M)
        if sign <= 0:
            raise RuntimeError("Encountered a non-positive determinant while solving theta_star(lambda).")
        objective += lam[s] * (-logdet)
        M_inv = np.linalg.inv(M)
        XM_inv = X @ M_inv
        diag_A_s = np.sum(XM_inv * X, axis=1)
        grad_theta += lam[s] * (-diag_A_s)
    return float(objective), grad_theta


def solve_theta_star_for_lambda(lam, max_iters=250, tol=1e-8):
    theta = np.full(d, 1.0 / d)
    objective, grad = d_optimal_objective_and_grad_theta(theta, lam)
    step = 1.0

    for _ in range(max_iters):
        projected_grad = theta - project_to_simplex(theta - grad)
        if np.linalg.norm(projected_grad, ord=np.inf) < tol:
            break

        accepted = False
        trial_step = step
        for _ in range(30):
            candidate = project_to_simplex(theta - trial_step * grad)
            candidate_objective, candidate_grad = d_optimal_objective_and_grad_theta(candidate, lam)
            if candidate_objective <= objective - 1e-4 * np.linalg.norm(candidate - theta) ** 2:
                accepted = True
                break
            trial_step *= 0.5

        if not accepted:
            break

        if np.linalg.norm(candidate - theta, ord=1) < tol:
            theta = candidate
            objective = candidate_objective
            grad = candidate_grad
            break

        theta = candidate
        objective = candidate_objective
        grad = candidate_grad
        step = min(2.0 * trial_step, 1.0)

    return theta, objective


def fixed_beta_gap(beta, lam, theta_max_iters=250):
    lam = project_to_simplex(lam)
    theta_hat, _, _ = theta_from(beta, lam)
    learned_objective, _ = d_optimal_objective_and_grad_theta(theta_hat, lam)
    _, oracle_objective = solve_theta_star_for_lambda(lam, max_iters=theta_max_iters)
    return float(learned_objective - oracle_objective)


def finite_difference_simplex_gradient(fun, lam, eps=1e-5):
    grad = np.zeros_like(lam, dtype=float)
    for j in range(len(lam)):
        direction = np.zeros_like(lam, dtype=float)
        direction[j] = 1.0
        plus = project_to_simplex(lam + eps * direction)
        minus = project_to_simplex(lam - eps * direction)
        denom = np.linalg.norm(plus - minus)
        if denom <= 0:
            continue
        grad[j] = (fun(plus) - fun(minus)) / denom
    return grad


def maximize_worst_case_gap_for_beta(
    beta,
    seed=0,
    num_lambda_starts=32,
    num_local_starts=6,
    outer_max_iters=40,
    theta_max_iters=250,
    use_ipopt=True,
):
    start_time = perf_counter()
    rng_gap = np.random.default_rng(seed)
    starts = [np.full(S, 1.0 / S)] + [np.eye(S)[j] for j in range(S)]
    starts.extend(rng_gap.dirichlet(np.ones(S), size=max(0, num_lambda_starts - len(starts))))

    objective_evaluations = 0

    def counted_gap(lam):
        nonlocal objective_evaluations
        objective_evaluations += 1
        return fixed_beta_gap(beta, lam, theta_max_iters=theta_max_iters)

    start_values = np.array([counted_gap(lam) for lam in starts])
    top_indices = np.argsort(start_values)[-min(num_local_starts, len(starts)):]
    best_idx = int(np.argmax(start_values))
    best_lam = np.array(starts[best_idx], dtype=float)
    best_gap = float(start_values[best_idx])
    local_statuses = []

    if use_ipopt:
        try:
            from cyipopt import minimize_ipopt
        except ImportError:
            local_statuses.append("cyipopt_unavailable")
        else:
            constraints = {
                "type": "eq",
                "fun": lambda lam: np.array([np.sum(lam) - 1.0]),
                "jac": lambda lam: np.ones((1, S)),
            }
            bounds = [(0.0, 1.0)] * S
            for idx in top_indices[::-1]:
                start_lam = np.array(starts[int(idx)], dtype=float)
                try:
                    result = minimize_ipopt(
                        lambda lam: -counted_gap(lam),
                        start_lam,
                        jac=lambda lam: -finite_difference_simplex_gradient(counted_gap, project_to_simplex(lam)),
                        bounds=bounds,
                        constraints=constraints,
                        options={"max_iter": outer_max_iters, "tol": 1e-7, "print_level": 0},
                    )
                    candidate_lam = project_to_simplex(result.x)
                    candidate_gap = counted_gap(candidate_lam)
                    local_statuses.append(f"ipopt_success={bool(result.success)}")
                    if candidate_gap > best_gap:
                        best_gap = float(candidate_gap)
                        best_lam = candidate_lam.copy()
                except Exception as exc:
                    local_statuses.append(f"ipopt_error={type(exc).__name__}")
            return {
                "worst_case_gap": float(best_gap),
                "lambda": project_to_simplex(best_lam),
                "optimizer_method": "cyipopt_multistart_finite_difference",
                "optimizer_status": ";".join(local_statuses),
                "objective_evaluations": int(objective_evaluations),
                "elapsed_seconds": perf_counter() - start_time,
            }

    for idx in top_indices[::-1]:
        lam = np.array(starts[int(idx)], dtype=float)
        value = float(start_values[int(idx)])
        step = 0.25
        improved_steps = 0

        for _ in range(outer_max_iters):
            grad = finite_difference_simplex_gradient(counted_gap, lam)
            if np.linalg.norm(grad, ord=np.inf) < 1e-8:
                break

            accepted = False
            trial_step = step
            for _ in range(20):
                candidate = project_to_simplex(lam + trial_step * grad)
                candidate_value = counted_gap(candidate)
                if candidate_value > value + 1e-8:
                    accepted = True
                    break
                trial_step *= 0.5

            if not accepted:
                break

            lam = candidate
            value = float(candidate_value)
            step = min(2.0 * trial_step, 1.0)
            improved_steps += 1

        local_statuses.append(f"improved_steps={improved_steps}")
        if value > best_gap:
            best_gap = value
            best_lam = lam.copy()

    return {
        "worst_case_gap": float(best_gap),
        "lambda": project_to_simplex(best_lam),
        "optimizer_method": "multistart_projected_gradient_finite_difference",
        "optimizer_status": ";".join(local_statuses),
        "objective_evaluations": int(objective_evaluations),
        "elapsed_seconds": perf_counter() - start_time,
    }


def compute_worst_case_gaps_from_snapshots(
    snapshots,
    checkpoint_interval=100,
    num_lambda_starts=32,
    num_local_starts=6,
    outer_max_iters=40,
    theta_max_iters=250,
    seed=SEED + 8675309,
    use_ipopt=True,
):
    rows = []
    for snapshot_index, snapshot in enumerate(snapshots):
        result = maximize_worst_case_gap_for_beta(
            snapshot["beta"],
            seed=seed + snapshot_index,
            num_lambda_starts=num_lambda_starts,
            num_local_starts=num_local_starts,
            outer_max_iters=outer_max_iters,
            theta_max_iters=theta_max_iters,
            use_ipopt=use_ipopt,
        )
        row = {
            "experiment": snapshot.get("experiment", f"fixed gamma={snapshot['gamma']:g}"),
            "mode": snapshot.get("mode", "fixed"),
            "gamma": snapshot["gamma"],
            "epoch_index": snapshot["epoch_index"],
            "phase": snapshot.get("phase", np.nan),
            "phase_iteration": snapshot.get("phase_iteration", np.nan),
            "snapshot_reason": snapshot.get("snapshot_reason", "checkpoint"),
            "previous_gamma": snapshot.get("previous_gamma", np.nan),
            "checkpoint_interval": checkpoint_interval,
            "snapshot_index": snapshot_index,
            "worst_case_gap": result["worst_case_gap"],
            "optimizer_method": result["optimizer_method"],
            "optimizer_status": result["optimizer_status"],
            "objective_evaluations": result["objective_evaluations"],
            "elapsed_seconds": result["elapsed_seconds"],
        }
        for j, value in enumerate(result["lambda"], start=1):
            row[f"lambda_{j}"] = float(value)
        rows.append(row)
    return pd.DataFrame(rows)


def merge_fixed_gamma_results(existing_results, new_results, gamma_tol=1e-12):
    """Append new gamma histories and replace existing histories for repeated gammas."""
    if new_results.empty:
        return existing_results.copy()

    if existing_results is None or existing_results.empty:
        merged = new_results.copy()
    else:
        new_gammas = new_results["gamma"].dropna().to_numpy(dtype=float)
        existing_gammas = existing_results["gamma"].to_numpy(dtype=float)
        repeated_mask = np.zeros(len(existing_results), dtype=bool)
        for gamma_value in new_gammas:
            repeated_mask |= np.isclose(existing_gammas, gamma_value, rtol=0.0, atol=gamma_tol)
        merged = pd.concat(
            [existing_results.loc[~repeated_mask].copy(), new_results.copy()],
            ignore_index=True,
        )

    return merged.sort_values(["gamma", "t"], kind="stable").reset_index(drop=True)


def rolling_average_difference(values):
    values = list(values)
    if len(values) < 2:
        raise ValueError("Need at least two values for adjacent rolling-average comparison.")
    previous_average = np.mean(values[:-1], axis=0)
    current_average = np.mean(values[1:], axis=0)
    difference = current_average - previous_average
    return float(np.linalg.norm(difference))


def detector_sample(beta, grad_f, no_reg_obj, metric):
    if metric == "gradient":
        return np.asarray(grad_f, dtype=float).reshape(-1)
    if metric == "objective":
        return np.array([float(no_reg_obj)], dtype=float)
    raise ValueError("adaptive saturation metric must be 'gradient' or 'objective'.")


def run_adaptive_gamma_experiment(
    beta_init,
    gamma_start=1.0,
    gamma_shrink=0.1,
    saturation_metric="objective",
    saturation_threshold=1e-3,
    saturation_threshold_alpha=0.5,
    saturation_window=5,
    max_iters=None,
    checkpoint_interval=100,
):
    if gamma_start <= 0:
        raise ValueError("gamma_start must be positive.")
    if gamma_shrink <= 0:
        raise ValueError("gamma_shrink must be positive.")
    if saturation_window < 1:
        raise ValueError("saturation_window must be a positive integer.")
    if saturation_threshold < 0:
        raise ValueError("saturation_threshold must be nonnegative.")
    if saturation_threshold_alpha < 0:
        raise ValueError("saturation_threshold_alpha must be nonnegative.")
    if saturation_metric not in {"gradient", "objective"}:
        raise ValueError("saturation_metric must be 'gradient' or 'objective'.")

    beta = beta_init.copy()
    current_gamma = float(gamma_start)
    max_iters = T if max_iters is None else int(max_iters)
    detector_window = []
    phase = 0
    phase_iteration = 0
    rows = []
    snapshots = []

    def record_adaptive_snapshot(
        epoch_index,
        beta_value,
        gamma_value,
        phase_value,
        phase_iteration_value,
        snapshot_reason,
        saturated=False,
        previous_gamma=np.nan,
    ):
        snapshots.append({
            "experiment": "adaptive gamma",
            "mode": "adaptive",
            "gamma": float(gamma_value),
            "previous_gamma": float(previous_gamma) if np.isfinite(previous_gamma) else np.nan,
            "phase": phase_value,
            "phase_iteration": phase_iteration_value,
            "epoch_index": epoch_index,
            "snapshot_reason": snapshot_reason,
            "beta": beta_value.copy(),
            "saturated": saturated,
        })

    record_adaptive_snapshot(
        0,
        beta,
        current_gamma,
        phase,
        phase_iteration,
        snapshot_reason="initial",
    )
    rng_adaptive = np.random.default_rng(SEED + 2026)

    for t_global in range(max_iters):
        lam_t = sample_lambda(rng_adaptive)
        gamma_used = current_gamma
        phase_used = phase
        phase_iteration_used = phase_iteration
        sample_loss, grad_f = sample_loss_and_grad(beta, lam_t)
        no_reg_obj, reg_obj = estimated_stochastic_objective_for_gamma(beta, gamma_used)
        detector_window.append(detector_sample(beta, grad_f, no_reg_obj, saturation_metric))
        if len(detector_window) > saturation_window + 1:
            detector_window.pop(0)
        current_saturation_threshold = saturation_threshold * (gamma_used / gamma_start) ** saturation_threshold_alpha

        saturation_value = np.nan
        saturated = False
        if len(detector_window) == saturation_window + 1:
            saturation_value = rolling_average_difference(detector_window)
            saturated = saturation_value <= current_saturation_threshold

        eta_value = eta_for_gamma(gamma_used)
        rows.append({
            "experiment": "adaptive gamma",
            "mode": "adaptive",
            "gamma": gamma_used,
            "phase": phase_used,
            "phase_iteration": phase_iteration_used,
            "t": t_global,
            "objective_estimate": no_reg_obj,
            "reg_objective_estimate": reg_obj,
            "sample_loss": sample_loss,
            "eta": eta_value,
            "beta_min": beta.min(),
            "beta_max": beta.max(),
            "saturation_metric": saturation_metric,
            "saturation_value": saturation_value,
            "saturation_threshold": current_saturation_threshold,
            "saturation_threshold_ref": saturation_threshold,
            "saturation_threshold_alpha": saturation_threshold_alpha,
            "saturated": saturated,
        })

        beta = mirror_step_for_gamma(beta, grad_f, gamma_used)
        completed_epochs = t_global + 1

        if saturated:
            next_gamma = gamma_used * gamma_shrink
            next_phase = phase_used + 1
            next_phase_iteration = 0
            snapshot_reason = "phase_change"
            detector_window.clear()
        else:
            next_gamma = gamma_used
            next_phase = phase_used
            next_phase_iteration = phase_iteration_used + 1
            snapshot_reason = ""

        if checkpoint_due(completed_epochs, max_iters, checkpoint_interval):
            snapshot_reason = "checkpoint" if not snapshot_reason else f"checkpoint,{snapshot_reason}"

        if snapshot_reason:
            record_adaptive_snapshot(
                completed_epochs,
                beta,
                next_gamma,
                next_phase,
                next_phase_iteration,
                snapshot_reason=snapshot_reason,
                saturated=saturated,
                previous_gamma=gamma_used if saturated else np.nan,
            )

        current_gamma = next_gamma
        phase = next_phase
        phase_iteration = next_phase_iteration

    return pd.DataFrame(rows), snapshots


In [ ]:
def random_simplex_point(p, rng=None):
    x = rng.exponential(scale=1.0, size=p)
    return x / x.sum()

In [ ]:
# Run the requested fixed-gamma experiments and merge them into the saved CSV.
# New gamma values are appended; repeated gamma values replace their previous histories.
# Beta snapshots are saved locally for the separate worst-case gap cell below.
# # Initialize at the centroid
# beta_init = np.full(p, 1.0 / p)
# Initialize at a vertex
beta_init = np.full(p, 1e-3)
beta_init[0] = 1 - p * 1e-3

In [ ]:


rng_fixed = np.random.default_rng(SEED + 314159)
# beta_init = random_simplex_point(p, rng_fixed) # initialize randomly
fixed_lambda_sequence = [sample_lambda(rng_fixed) for _ in range(T)]

fixed_gamma_outputs = [
    run_fixed_gamma_experiment(
        gamma_value,
        beta_init,
        fixed_lambda_sequence,
        checkpoint_interval=checkpoint_interval,
    )
    for gamma_value in gamma_values
]

new_gamma_sweep_results = pd.concat(
    [output[0] for output in fixed_gamma_outputs],
    ignore_index=True,
) if len(fixed_gamma_outputs) > 0 else pd.DataFrame()

fixed_gamma_snapshots = [
    snapshot
    for _, snapshots in fixed_gamma_outputs
    for snapshot in snapshots
]
snapshot_path = save_fixed_gamma_beta_snapshots(fixed_gamma_snapshots, gamma_sweep_beta_snapshots_npz)

try:
    existing_gamma_sweep_results = pd.read_csv(gamma_sweep_csv)
except FileNotFoundError:
    existing_gamma_sweep_results = pd.DataFrame()

gamma_sweep_results = merge_fixed_gamma_results(
    existing_gamma_sweep_results,
    new_gamma_sweep_results,
)
gamma_sweep_results.to_csv(gamma_sweep_csv, index=False)

pd.DataFrame(
    {
        "trained_gamma_this_run": [float(gamma_value) for gamma_value in gamma_values],
        "status": [
            "replaced existing history"
            if (
                not existing_gamma_sweep_results.empty
                and np.isclose(
                    existing_gamma_sweep_results["gamma"].to_numpy(dtype=float),
                    float(gamma_value),
                    rtol=0.0,
                    atol=1e-12,
                ).any()
            )
            else "added new history"
            for gamma_value in gamma_values
        ],
        "beta_snapshot_file": str(snapshot_path),
        "num_beta_snapshots": len(fixed_gamma_snapshots),
    }
)


In [ ]:
# Reload the fixed-gamma beta snapshots and compute worst-case gap estimates.
reloaded_fixed_gamma_snapshots = load_fixed_gamma_beta_snapshots(gamma_sweep_beta_snapshots_npz)

if worst_case_gap_enabled:
    fixed_gamma_worst_case_gaps = compute_worst_case_gaps_from_snapshots(
        reloaded_fixed_gamma_snapshots,
        checkpoint_interval=checkpoint_interval,
        num_lambda_starts=worst_case_num_lambda_starts,
        num_local_starts=worst_case_num_local_starts,
        outer_max_iters=worst_case_outer_max_iters,
        theta_max_iters=worst_case_theta_max_iters,
        use_ipopt=use_ipopt_for_worst_case_gap,
    )
    fixed_gamma_worst_case_gaps.to_csv(gamma_sweep_worst_case_gap_csv, index=False)
else:
    fixed_gamma_worst_case_gaps = pd.DataFrame()

fixed_gamma_worst_case_gaps.head()


```
threshold = adaptive_saturation_threshold_ref * (
    current_gamma / adaptive_gamma_start
) ** adaptive_saturation_threshold_alpha
```

alpha = 0: fixed threshold, no gamma scaling.

alpha = 0.5: moderate scaling, good first choice.

alpha = 1.0: stronger scaling, threshold shrinks in direct proportion to gamma.

alpha > 1.0: very aggressive tightening as gamma gets smaller.

In [ ]:
# Run the adaptive-gamma experiment and save it separately.
adaptive_gamma_start = 1.0
adaptive_gamma_shrink = 0.1
adaptive_saturation_metric = "gradient"  # "objective" or "gradient"
adaptive_saturation_threshold_ref = 100 # 100 if gradient, 0.1 if objective
adaptive_saturation_threshold_alpha = .5
adaptive_saturation_window = 3

adaptive_gamma_results, adaptive_gamma_snapshots = run_adaptive_gamma_experiment(
    beta_init,
    gamma_start=adaptive_gamma_start,
    gamma_shrink=adaptive_gamma_shrink,
    saturation_metric=adaptive_saturation_metric,
    saturation_threshold=adaptive_saturation_threshold_ref,
    saturation_threshold_alpha=adaptive_saturation_threshold_alpha,
    saturation_window=adaptive_saturation_window,
    max_iters=T,
    checkpoint_interval=checkpoint_interval,
)
adaptive_gamma_results.to_csv(adaptive_gamma_csv, index=False)
adaptive_snapshot_path = save_fixed_gamma_beta_snapshots(adaptive_gamma_snapshots, adaptive_gamma_beta_snapshots_npz)
adaptive_gamma_results.head()


In [ ]:
# Compute worst-case gap estimates for the adaptive-gamma snapshots.
if worst_case_gap_enabled:
    adaptive_gamma_worst_case_gaps = compute_worst_case_gaps_from_snapshots(
        adaptive_gamma_snapshots,
        checkpoint_interval=checkpoint_interval,
        num_lambda_starts=worst_case_num_lambda_starts,
        num_local_starts=worst_case_num_local_starts,
        outer_max_iters=worst_case_outer_max_iters,
        theta_max_iters=worst_case_theta_max_iters,
        use_ipopt=use_ipopt_for_worst_case_gap,
    )
    adaptive_gamma_worst_case_gaps.to_csv(adaptive_gamma_worst_case_gap_csv, index=False)
else:
    adaptive_gamma_worst_case_gaps = pd.DataFrame()

adaptive_gamma_worst_case_gaps.head()


In [ ]:
# Summarize the saved downstream experiment results without plotting.
comparison_results = pd.concat([gamma_sweep_results, adaptive_gamma_results], ignore_index=True)

comparison_results.groupby("experiment")["reg_objective_estimate"].agg(["first", "last", "min"])


## 4. Mean-$\lambda$ Benchmark Line

Because `h(\theta, \lambda)` is linear in `\lambda`, replacing `\lambda` by its mean gives a deterministic weighted objective in `\theta`. This section now uses a Newton-type method, specifically a limited-memory BFGS solver on a softmax parameterization of `\theta`, to compute a sharper deterministic benchmark. The resulting best attained value is plotted as a grey horizontal reference line.


In [ ]:
# Deterministic theta-space benchmark using a Newton-type solver (L-BFGS) with mean lambda.
theta_reference_csv = "theta_mean_lambda_reference_results.csv"
lambda_mean = np.full(S, 1.0 / S)


def softmax_theta(z):
    z_shifted = z - np.max(z)
    exp_z = np.exp(z_shifted)
    return exp_z / exp_z.sum()


def mean_lambda_theta_objective_and_grad(theta):
    objective = 0.0
    grad_theta = np.zeros(d)
    for s, X in enumerate(X_list):
        M = information_matrix(X, theta)
        sign, logdet = np.linalg.slogdet(M)
        if sign <= 0:
            raise RuntimeError("Encountered a non-positive determinant in the theta-space mean-lambda benchmark.")
        objective += lambda_mean[s] * (-logdet)
        M_inv = np.linalg.inv(M)
        XM_inv = X @ M_inv
        diag_A_s = np.sum(XM_inv * X, axis=1)
        grad_theta += lambda_mean[s] * (-diag_A_s)
    return float(objective), grad_theta


def theta_objective_from_z(z):
    theta = softmax_theta(z)
    objective, grad_theta = mean_lambda_theta_objective_and_grad(theta)
    centered_grad = grad_theta - np.dot(theta, grad_theta)
    grad_z = theta * centered_grad
    return objective, grad_z, theta


def lbfgs_direction(grad, s_hist, y_hist):
    if not s_hist:
        return -grad

    q_vec = grad.copy()
    alpha_vals = []
    rho_vals = []

    for s_vec, y_vec in zip(reversed(s_hist), reversed(y_hist)):
        rho = 1.0 / np.dot(y_vec, s_vec)
        alpha = rho * np.dot(s_vec, q_vec)
        rho_vals.append(rho)
        alpha_vals.append(alpha)
        q_vec = q_vec - alpha * y_vec

    last_s = s_hist[-1]
    last_y = y_hist[-1]
    scale = np.dot(last_s, last_y) / np.dot(last_y, last_y)
    r_vec = scale * q_vec

    for s_vec, y_vec, rho, alpha in zip(s_hist, y_hist, reversed(rho_vals), reversed(alpha_vals)):
        beta = rho * np.dot(y_vec, r_vec)
        r_vec = r_vec + s_vec * (alpha - beta)

    return -r_vec


def run_mean_lambda_lbfgs(max_iters=80, memory=8, c1=1e-4, backtrack=0.5):
    z = np.zeros(d)
    objective, grad_z, theta = theta_objective_from_z(z)
    s_hist = []
    y_hist = []
    rows = []

    for t_theta in range(max_iters):
        grad_norm = float(np.linalg.norm(grad_z))
        rows.append({
            "t": t_theta,
            "mean_lambda_theta_objective": objective,
            "grad_norm": grad_norm,
        })

        direction = lbfgs_direction(grad_z, s_hist, y_hist)
        if np.dot(direction, grad_z) >= 0:
            direction = -grad_z

        step = 1.0
        while step > 1e-8:
            z_candidate = z + step * direction
            objective_candidate, grad_candidate, theta_candidate = theta_objective_from_z(z_candidate)
            if objective_candidate <= objective + c1 * step * np.dot(grad_z, direction):
                break
            step *= backtrack

        if step <= 1e-8:
            break

        s_vec = z_candidate - z
        y_vec = grad_candidate - grad_z
        sy = float(np.dot(s_vec, y_vec))
        if sy > 1e-12:
            s_hist.append(s_vec)
            y_hist.append(y_vec)
            if len(s_hist) > memory:
                s_hist.pop(0)
                y_hist.pop(0)

        z = z_candidate
        objective = objective_candidate
        grad_z = grad_candidate
        theta = theta_candidate

        if grad_norm < 1e-7:
            rows.append({
                "t": t_theta + 1,
                "mean_lambda_theta_objective": objective,
                "grad_norm": float(np.linalg.norm(grad_z)),
            })
            break

    return pd.DataFrame(rows), theta, z


theta_reference_results, theta_reference, z_reference = run_mean_lambda_lbfgs(max_iters=80, memory=8)
theta_reference_results.to_csv(theta_reference_csv, index=False)
theta_reference_level = float(theta_reference_results["mean_lambda_theta_objective"].min())
theta_reference_results.head()


In [ ]:
theta_reference_level

## 5. Five-Point $\lambda$ Simplex Check

A very large regularization weight such as `\gamma = 10` pushes `\beta` toward the center of the `qd`-dimensional simplex. In this limit we fix `\beta` at the simplex center, evaluate the unregularized objective at the four corners of the 4-simplex for `\lambda`, and also at the centroid. We then compare those values against the deterministic mean-`\lambda` benchmark design `theta_reference`.


In [ ]:
# Compare the heavy-regularization beta-center design against the mean-lambda benchmark
# at the four lambda corners and the lambda centroid.

def objective_at_theta(theta, lam):
    objective = 0.0
    for s, X in enumerate(X_list):
        M = information_matrix(X, theta)
        sign, logdet = np.linalg.slogdet(M)
        if sign <= 0:
            raise RuntimeError("Encountered a non-positive determinant in lambda-point evaluation.")
        objective += lam[s] * (-logdet)
    return float(objective)


beta_center = np.full(p, 1.0 / p)

lambda_points = pd.DataFrame(
    {
        "lambda_label": [f"corner {i + 1}" for i in range(S)] + ["centroid"],
        "lambda": [np.eye(S)[i] for i in range(S)] + [np.full(S, 1.0 / S)],
    }
)

rows = []
for _, row in lambda_points.iterrows():
    lam = row["lambda"]
    theta_from_fixed_beta, _, _ = theta_from(beta_center, lam)
    fixed_beta_objective = objective_at_theta(theta_from_fixed_beta, lam)
    benchmark_objective = objective_at_theta(theta_reference, lam)

    rows.append(
        {
            "lambda_label": row["lambda_label"],
            "lambda": np.array2string(lam, precision=3, separator=", "),
            "fixed_beta_unregularized_objective": fixed_beta_objective,
            "benchmark_unregularized_objective": benchmark_objective,
            "gap_fixed_minus_benchmark": fixed_beta_objective - benchmark_objective,
        }
    )

lambda_point_comparison = pd.DataFrame(rows)
lambda_point_comparison_csv = "lambda_point_comparison.csv"
lambda_point_comparison.to_csv(lambda_point_comparison_csv, index=False)
lambda_point_comparison
